# Phase 4 v0.5 Round — Kaggle CUDA

**실행 전 체크리스트**
- 우측 **Data** 탭 → **Add data** → `donghyun51/lens-phase4-v0-4` 추가
- Accelerator: **GPU T4 x1 이상**
- Internet: **On**


## Cell 1 — GPU 확인


In [ ]:
import subprocess, torch
print(subprocess.check_output(['nvidia-smi'], text=True))
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## Cell 2 — Input 경로 진단

`/kaggle/input/` 아래 실제 구조를 확인하고 파일을 찾는다.


In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')

# /kaggle/input/ 아래 최상위 디렉토리 목록 출력
print('=== /kaggle/input/ 구조 ===')
if INPUT_ROOT.exists():
    for d in sorted(INPUT_ROOT.iterdir()):
        files = list(d.glob('*'))
        print(f'  {d.name}/')
        for f in sorted(files)[:6]:  # 최대 6개만
            size_mb = f.stat().st_size / 1e6 if f.is_file() else 0
            print(f'    {f.name}  ({size_mb:.1f} MB)' if f.is_file() else f'    {f.name}/')
else:
    print('  /kaggle/input 없음 — 로컬 실행 중?')


## Cell 3 — 파일 경로 자동 설정

위 Cell 2 출력에서 마운트된 폴더명을 확인한 뒤,
아래 `DATASET_SLUG`를 실제 폴더명으로 수정한다 (기본값: `lens-phase4-v0-4`).


In [ ]:
from pathlib import Path

# Cell 2 출력에서 확인한 폴더명으로 수정 (기본값: lens-phase4-v0-4)
DATASET_SLUG = 'lens-phase4-v0-4'

INPUT_DIR  = Path('/kaggle/input') / DATASET_SLUG
WORK_DIR   = Path('/kaggle/working')

# ── 파일 경로 후보 탐색 (슬러그가 다를 때 자동 탐색) ────────────────────────
def find_file(name: str) -> Path | None:
    # 1순위: 지정 슬러그 폴더
    p = INPUT_DIR / name
    if p.exists(): return p
    # 2순위: /kaggle/input/ 전체 탐색
    matches = list(Path('/kaggle/input').rglob(name))
    return matches[0] if matches else None

TRAIN_H5   = find_file('phase4_v0_4.h5')
UNFILT_H5  = find_file('phase4_v0_4_eval_unfiltered.h5')
SCALER_PKL = find_file('target_scaler_phase4_v0_4.pkl')

print('TRAIN_H5  :', TRAIN_H5)
print('UNFILT_H5 :', UNFILT_H5)
print('SCALER_PKL:', SCALER_PKL)

assert TRAIN_H5,   'phase4_v0_4.h5 를 찾을 수 없음 — Datasets에서 donghyun51/lens-phase4-v0-4 추가 확인'
assert UNFILT_H5,  'phase4_v0_4_eval_unfiltered.h5 를 찾을 수 없음'
assert SCALER_PKL, 'target_scaler_phase4_v0_4.pkl 를 찾을 수 없음'

print('\nAll input files found OK')
for p in [TRAIN_H5, UNFILT_H5, SCALER_PKL]:
    print(f'  {p.name}: {p.stat().st_size/1e6:.1f} MB')


## Cell 4 — 코드 클론 (GitHub main)


In [ ]:
import subprocess

REPO_URL = 'https://github.com/dasbaq/GV.git'
REPO_ROOT = Path('/kaggle/working/repo')
PROJ_DIR  = REPO_ROOT / 'Gravitational_Lens_MultiMode'

if not REPO_ROOT.exists():
    print('Cloning...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)
else:
    print('Repo exists, pulling...')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

# v0.5 핵심 파일 확인
checks = {
    'v0.5 round script': PROJ_DIR / 'scripts/phase4_v0_5_round.py',
    'round_eval':        PROJ_DIR / 'ml/training/round_eval.py',
    'physics_pairing':   PROJ_DIR / 'ml/training/physics_pairing.py',
    'heads.py':          PROJ_DIR / 'ml/models/heads.py',
    'mode3_wrapper GONE': None,  # 아래에서 별도 확인
}
for label, path in checks.items():
    if path is None:
        gone = not (PROJ_DIR / 'inversion/mode3_wrapper.py').exists()
        print(f'  {"✅" if gone else "❌"} {label}: {"deleted" if gone else "STILL EXISTS"}')
    else:
        print(f'  {"✅" if path.exists() else "❌"} {label}')

# Mode1Head in_dim 확인 (256 = v0.5, 384 = v0.4)
heads_src = (PROJ_DIR / 'ml/models/heads.py').read_text()
if 'd_model * 2' in heads_src or 'in_dim=d_model * 2' in heads_src or 'in_dim: int = 256' in heads_src:
    print('  ✅ Mode1Head in_dim = d_model×2 (v0.5)')
elif '384' in heads_src or 'd_model * 3' in heads_src:
    print('  ❌ Mode1Head still d_model×3 — v0.4 코드가 clone됨, push 확인 필요')
else:
    print('  ⚠️  Mode1Head in_dim 확인 불명 — heads.py 수동 확인')


## Cell 5 — 환경변수 + import 검증


In [ ]:
import os, sys

os.environ['LENS_DATA_PATH']            = str(TRAIN_H5)
os.environ['LENS_DATA_PATH_UNFILTERED'] = str(UNFILT_H5)
os.environ['LENS_SCALER_PATH']          = str(SCALER_PKL)
os.environ['LENS_WORK_ROOT']            = str(WORK_DIR)

os.chdir(str(PROJ_DIR))
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print('cwd:', os.getcwd())
for k in ['LENS_DATA_PATH','LENS_DATA_PATH_UNFILTERED','LENS_SCALER_PATH','LENS_WORK_ROOT']:
    print(f'  {k} = {os.environ[k]}')

subprocess.run(['pip', 'install', '-q', 'h5py', 'scipy', 'pyyaml'], check=True)

from ml.models.error_corrector import MultiModalErrorCorrector
from ml.training.round_eval    import evaluate_mode1_h0_on_loader
from ml.training.physics_pairing import add_paired_physics_predictions
print('All imports OK ✅')


## Cell 6 — 2-epoch sanity run

NaN 없이 완료되면 Full run 진행.


In [ ]:
ROUND_SCRIPT = str(PROJ_DIR / 'scripts' / 'phase4_v0_5_round.py')
!python {ROUND_SCRIPT} --phase train --device cuda --workers 0 --epochs 2 --bootstrap-n 0


## Cell 7 — Full run (50 epochs)

sanity에서 NaN이 없으면 실행. 예상 시간: ~30-60분 (T4 기준).


In [ ]:
!python {ROUND_SCRIPT} --phase train --device cuda --workers 0 --epochs 50 --bootstrap-n 1000


## Cell 8 — 결과 확인 (acceptance report)


In [ ]:
import json
logs_dir = WORK_DIR / 'logs'

for fname in ['phase4_v0_5_imgres_h0_eval.json',
              'phase4_v0_5_imgres_h0_eval_unfiltered.json',
              'phase4_v0_5_infra_equivalence.json']:
    p = logs_dir / fname
    if not p.exists(): print(f'⏳ {fname}: not found yet'); continue
    data = json.loads(p.read_text())
    print(f'\n== {fname} ==')
    if 'stage_b_acceptance_report' in data:
        rep = data['stage_b_acceptance_report']
        print('all_pass:', rep.get('all_pass_excluding_record_only'),
              '| leak_triggered:', rep.get('leak_triggered'))
        for row in rep.get('pass_rows', []):
            mark = '✅' if row['pass'] else ('📝' if row['pass'] is None else '❌')
            print(f"  {mark} {row['metric']}: {row['value']}")
    elif 'best' in data:
        m   = data['best']['mode1']['h0']['model']
        cal = data['best']['mode1']['log_sigma_calibration']
        print(f"  RMSE={m['RMSE']:.3f}  r={m['r']:.4f}  coverage={cal['coverage_abs_residual_le_1sigma']:.3f}")


## Cell 9 — Checkpoint v0.5 검증

`head1.net.0.weight` shape `(64, 256)` = v0.5 OK  
`img_enc.*` / `head3.*` 키 없음 = Image/Mode3 삭제 확인


In [ ]:
import torch
ckpt_path = WORK_DIR / 'checkpoints' / 'phase4_v0_5_imgres_best.pt'

if not ckpt_path.exists():
    print(f'checkpoint not found: {ckpt_path}')
    print('Full run (Cell 7) 완료 후 다시 실행하세요.')
else:
    sd = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    w   = sd.get('head1.net.0.weight')
    par = sd.get('par_enc.net.0.weight')
    img_keys = [k for k in sd if k.startswith('img_enc.')]
    h3_keys  = [k for k in sd if k.startswith('head3.')]

    print(f'head1.net.0.weight : {w.shape}')     # 기대: (64, 256)
    print(f'par_enc.net.0.weight: {par.shape}')   # 기대: (256, 20)
    print(f'img_enc keys (기대 0): {len(img_keys)}')
    print(f'head3  keys (기대 0): {len(h3_keys)}')

    assert w.shape   == (64, 256), f'Expected (64,256), got {w.shape}'
    assert par.shape[1] == 20,     f'Expected par_enc 20-dim, got {par.shape}'
    assert len(img_keys) == 0,     f'img_enc keys found: {img_keys[:3]}'
    assert len(h3_keys)  == 0,     f'head3 keys found: {h3_keys[:3]}'
    print('\nv0.5 checkpoint verified ✅')


## Cell 10 — 산출물 목록 (회수 대상)


In [ ]:
artifacts = [
    WORK_DIR / 'checkpoints' / 'phase4_v0_5_imgres_best.pt',
    WORK_DIR / 'logs' / 'phase4_v0_5_imgres_h0_eval.json',
    WORK_DIR / 'logs' / 'phase4_v0_5_imgres_h0_eval_unfiltered.json',
    WORK_DIR / 'logs' / 'phase4_v0_5_imgres_long_history.json',
    WORK_DIR / 'logs' / 'phase4_v0_5_infra_equivalence.json',
]
print('== 회수 대상 산출물 (Kaggle output 탭에서 다운로드) ==')
for p in artifacts:
    size = f'{p.stat().st_size/1e6:.1f} MB' if p.exists() else '아직 없음'
    print(f'  {"✅" if p.exists() else "⏳"} {p.name}  ({size})')

print('\n== 로컬 저장 위치 ==')
print('  phase4_v0_5_imgres_best.pt         → data/checkpoints/')
print('  phase4_v0_5_imgres_*.json          → data/logs/')
